# 00 - Ortam ve Drive kontrolü
Bu notebook yerel Jupyter, VS Code + Colab ve tarayıcı Colab akışını doğrular.

In [ ]:
# Ortak bootstrap: kod Git'ten; Git URL'si yoksa Drive'daki kaynak paketinden gelir.
import os, sys, shutil, subprocess, zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    artifacts_root = Path('/content/drive/MyDrive/PhysioLoad_AI_Project')
    code_root = Path('/content/physioload-ai')
    repo_url = os.environ.get('PHYSIOLOAD_REPO_URL', '').strip()
    branch = os.environ.get('PHYSIOLOAD_REPO_BRANCH', 'develop')
    if repo_url:
        if (code_root / '.git').exists():
            subprocess.run(['git', '-C', str(code_root), 'fetch', '--depth', '1', 'origin', branch], check=True)
            subprocess.run(['git', '-C', str(code_root), 'reset', '--hard', f'origin/{branch}'], check=True)
        else:
            subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, repo_url, str(code_root)], check=True)
    else:
        archive = artifacts_root / 'code' / 'physioload-source.zip'
        if not archive.exists():
            raise FileNotFoundError(f'Kaynak paketi yok: {archive}')
        if code_root.exists():
            shutil.rmtree(code_root)
        code_root.mkdir(parents=True)
        with zipfile.ZipFile(archive) as package:
            package.extractall(code_root)
else:
    artifacts_root = None
    here = Path.cwd().resolve()
    code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())

sys.path.insert(0, str(code_root))
from physioload_env import setup_environment
env = setup_environment(root=code_root, artifacts_root=artifacts_root)
print(env.summary())

In [ ]:
from pathlib import Path
import importlib.util, platform

checks = {
    'python': platform.python_version(),
    'mediapipe': importlib.util.find_spec('mediapipe') is not None,
    'rtmlib': importlib.util.find_spec('rtmlib') is not None,
    'torch': importlib.util.find_spec('torch') is not None,
    'drive_writable': env.artifacts_root.exists(),
}
checks

In [ ]:
from physioload.geometry import joint_angle_degrees
from physioload.quality import QualityFeatures, QualityGate

angle = joint_angle_degrees((0, 0), (1, 0), (1, 1))
decision = QualityGate().evaluate(
    QualityFeatures(0.95, 0.95, 5.0, 0.05, True, True),
    requested_outputs=['joint_angle'],
)
print('smoke angle:', angle, 'decision:', decision)